# 📖 Notebook 1: Operational Transformation Basics

When two people edit the same document at the same time, their edits can **conflict**. Operational Transformation (OT) is the algorithm that resolves these conflicts so everyone sees the same final document.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why naive concurrent editing breaks documents
- What an "operation" is (insert/delete at a position)
- How OT transforms operations to preserve intent
- Why Google Docs chose OT over other approaches

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/google-docs
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `googledocs_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import json

# Database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "googledocs_demo",
    "user": "demo",
    "password": "demo"
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

# Test connection
try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker-compose up -d")

## 🤔 The Problem: Concurrent Edits Break Documents

Imagine Alice and Bob are both editing the word `Hello!`:

```
Starting document: "Hello!"
                    012345

Alice wants to add ", world" after "Hello"  →  INSERT(5, ", world")
Bob wants to delete the "!"                  →  DELETE(5, 1)
```

If we just apply both edits naively, the order they arrive matters:

- **Alice first, then Bob**: `Hello, world!` → delete at 5 → `Hello world!` ❌ (deleted the comma!)
- **Bob first, then Alice**: `Hello` → insert at 5 → `Hello, world` ✅

**Neither order gives us what both users intended!** Alice wanted `Hello, world` and Bob wanted the `!` removed.

The correct result should be: **`Hello, world`**

This is exactly the problem OT solves.

In [ ]:
# Let's see the problem in code

def apply_insert(text, position, content):
    """Insert `content` at `position` in the text."""
    return text[:position] + content + text[position:]

def apply_delete(text, position, length):
    """Delete `length` characters starting at `position`."""
    return text[:position] + text[position + length:]

original = "Hello!"
print(f"Original document: '{original}'")
print(f"Positions:          {''.join(str(i) for i in range(len(original)))}")
print()

# Alice's edit: INSERT ", world" at position 5
alice_op = {"type": "insert", "position": 5, "content": ", world"}

# Bob's edit: DELETE 1 char at position 5 (the "!")
bob_op = {"type": "delete", "position": 5, "length": 1}

print("Alice's intent: INSERT(5, ', world') — add ', world' after 'Hello'")
print("Bob's intent:   DELETE(5, 1) — remove the '!'")
print()

# Scenario 1: Alice first, then Bob (WRONG)
after_alice = apply_insert(original, 5, ", world")
after_both_1 = apply_delete(after_alice, 5, 1)  # Bob's DELETE(5,1) hits the comma!
print(f"Order 1 (Alice→Bob): '{original}' → '{after_alice}' → '{after_both_1}'")
print(f"  ❌ Bob deleted the comma instead of the '!'")
print()

# Scenario 2: Bob first, then Alice (ALSO WRONG — missing the '!' deletion intent)
after_bob = apply_delete(original, 5, 1)
after_both_2 = apply_insert(after_bob, 5, ", world")
print(f"Order 2 (Bob→Alice): '{original}' → '{after_bob}' → '{after_both_2}'")
print(f"  ✅ Happens to work, but only by luck of ordering!")
print()
print("💡 We need a way to adjust operations so they work regardless of order.")

## 🔧 What Is an Operation?

In OT, every edit to a document is represented as an **operation**:

| Operation | Fields | Example | Meaning |
|-----------|--------|---------|---------|
| **INSERT** | position, content | INSERT(5, ", world") | Insert ", world" at position 5 |
| **DELETE** | position, length | DELETE(5, 1) | Delete 1 character at position 5 |

Every operation is **contextual** — it was created while looking at a specific version of the document. When two users create operations based on the same version, those operations need to be **transformed** before one can be applied after the other.

In [ ]:
# Let's define operations as simple dictionaries

def make_insert(position, content):
    """Create an INSERT operation."""
    return {"type": "insert", "position": position, "content": content}

def make_delete(position, length=1):
    """Create a DELETE operation."""
    return {"type": "delete", "position": position, "length": length}

def apply_op(text, op):
    """Apply an operation to a text string."""
    if op["type"] == "insert":
        return apply_insert(text, op["position"], op["content"])
    elif op["type"] == "delete":
        return apply_delete(text, op["position"], op["length"])
    return text

# Demo: operations on a simple document
doc = ""
print(f"Start: '{doc}'")

ops = [
    make_insert(0, "Hello"),
    make_insert(5, "!"),
    make_insert(5, ", world"),
    make_delete(12, 1),  # remove the "!"
]

for i, op in enumerate(ops):
    doc = apply_op(doc, op)
    print(f"  Op {i+1}: {op['type'].upper()}({op['position']}, {op.get('content', op.get('length', ''))}) → '{doc}'")

print(f"\nFinal: '{doc}'")

## 🔄 The OT Algorithm: Transform Before Applying

The core idea of OT is simple:

> **Before applying an operation, transform it against all operations that have been applied since it was created.**

The transformation rules are intuitive:

### Rule 1: INSERT before INSERT
If operation A **inserts** text before operation B's position, B's position shifts **right** by the length of A's insertion.

### Rule 2: DELETE before INSERT
If operation A **deletes** text before operation B's position, B's position shifts **left** by the length of A's deletion.

### Rule 3: INSERT before DELETE
If operation A **inserts** text before operation B's delete position, B's position shifts **right**.

### Rule 4: DELETE before DELETE
If operation A **deletes** text before operation B's delete position, B's position shifts **left**.

```
Think of it like editing a numbered list:
If someone adds an item above yours, your item's number increases.
If someone removes an item above yours, your item's number decreases.
```

In [ ]:
def transform(op_a, op_b):
    """
    Transform operation B against operation A.
    
    Precondition: both ops were created against the same document state.
    A has already been applied. We need to adjust B so it still
    makes sense after A.
    
    Returns: the transformed version of op_b.
    """
    b = dict(op_b)  # copy so we don't modify the original
    
    if op_a["type"] == "insert":
        insert_len = len(op_a["content"])
        # A inserted text — if B is at or after A's position, shift B right
        if b["position"] >= op_a["position"]:
            b["position"] += insert_len
            
    elif op_a["type"] == "delete":
        delete_len = op_a.get("length", 1)
        if b["position"] > op_a["position"]:
            # B is after the deleted region — shift B left
            b["position"] = max(op_a["position"], b["position"] - delete_len)
    
    return b

print("OT Transform function defined! ✅")
print("Let's test it on our Alice & Bob example...")

In [ ]:
# Back to our example: Alice and Bob both edit "Hello!"
original = "Hello!"

alice_op = make_insert(5, ", world")  # insert ", world" at position 5
bob_op = make_delete(5, 1)             # delete 1 char at position 5 (the "!")

print(f"Original: '{original}'")
print(f"Alice: INSERT(5, ', world')")
print(f"Bob:   DELETE(5, 1)")
print()

# The server receives Alice's op first, applies it
server_doc = apply_op(original, alice_op)
print(f"Server applies Alice's op: '{server_doc}'")

# Now the server needs to apply Bob's op — but it was created against the OLD document!
# We transform Bob's op against Alice's op
bob_transformed = transform(alice_op, bob_op)
print(f"\nBob's original op:     DELETE({bob_op['position']}, {bob_op['length']})")
print(f"Bob's transformed op:  DELETE({bob_transformed['position']}, {bob_transformed['length']})")
print(f"  (shifted right by {len(alice_op['content'])} because Alice inserted {len(alice_op['content'])} chars before position 5)")

# Apply the transformed op
final = apply_op(server_doc, bob_transformed)
print(f"\nFinal document: '{final}'")
print(f"\n✅ Both intents preserved: Alice's ', world' is there, and Bob's '!' is removed!")

## 🧪 More OT Examples

Let's test more scenarios to build intuition for how OT handles different cases.

In [ ]:
def simulate_concurrent_edit(original, op_a, op_b, name_a="User A", name_b="User B"):
    """
    Simulate two users making concurrent edits and show how OT resolves them.
    
    Both ops were created against the same `original` document.
    We apply op_a first, then transform and apply op_b.
    """
    print(f"Original: '{original}'")
    op_a_desc = f"{op_a['type'].upper()}({op_a['position']}, {op_a.get('content', op_a.get('length', ''))})"
    op_b_desc = f"{op_b['type'].upper()}({op_b['position']}, {op_b.get('content', op_b.get('length', ''))})"
    print(f"{name_a}: {op_a_desc}")
    print(f"{name_b}: {op_b_desc}")
    print()
    
    # Apply A first
    after_a = apply_op(original, op_a)
    print(f"After {name_a}: '{after_a}'")
    
    # Transform B against A, then apply
    transformed_b = transform(op_a, op_b)
    tb_desc = f"{transformed_b['type'].upper()}({transformed_b['position']}, {transformed_b.get('content', transformed_b.get('length', ''))})"
    print(f"{name_b} transformed: {tb_desc}")
    
    final = apply_op(after_a, transformed_b)
    print(f"Final: '{final}'")
    print()
    return final


# Example 1: Two inserts at different positions
print("═" * 60)
print("Example 1: Two inserts at different positions")
print("═" * 60)
simulate_concurrent_edit(
    "Hello",
    make_insert(0, "Oh, "),      # User A adds "Oh, " at the start
    make_insert(5, " world"),    # User B adds " world" at the end
    "Alice", "Bob"
)

# Example 2: Two inserts at the SAME position
print("═" * 60)
print("Example 2: Two inserts at the same position")
print("═" * 60)
simulate_concurrent_edit(
    "Hello world",
    make_insert(5, " beautiful"),  # Alice adds "beautiful" after "Hello"
    make_insert(5, " wonderful"),  # Bob also adds at position 5
    "Alice", "Bob"
)

# Example 3: Delete then insert
print("═" * 60)
print("Example 3: Delete before insert")
print("═" * 60)
simulate_concurrent_edit(
    "Hello world!",
    make_delete(5, 6),            # Alice deletes " world" (6 chars at position 5)
    make_insert(12, " :)"),       # Bob adds smiley at the end
    "Alice", "Bob"
)

## 📝 Seeing OT in the Database

Our doc server stores every operation in PostgreSQL. Let's look at the operations that built our seed documents.

In [ ]:
# Query the operations table
conn = get_db()
cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cursor.execute("""
    SELECT o.id, o.op_type, o.position, o.content, o.length,
           u.display_name AS user_name
    FROM operations o
    JOIN users u ON o.user_id = u.id
    WHERE o.document_id = 1
    ORDER BY o.id
""")
ops = cursor.fetchall()
conn.close()

print("📜 Operations log for Document 1 (Meeting Notes):")
print(f"{'ID':>3}  {'User':<16} {'Type':<8} {'Pos':>4}  Content")
print("-" * 70)
for op in ops:
    content = op['content'][:40] + '...' if len(op['content'] or '') > 40 else op['content']
    print(f"{op['id']:>3}  {op['user_name']:<16} {op['op_type']:<8} {op['position']:>4}  {content}")

print(f"\n💡 Notice how Alice, Bob, and Charlie all contributed operations!")
print(f"   Each operation was applied sequentially by the server using OT.")

In [ ]:
# Let's replay the operations to rebuild the document from scratch

doc = ""  # start empty
print("🔄 Replaying operations from the database:")
print()

for op in ops:
    op_dict = {
        "type": op["op_type"],
        "position": op["position"],
        "content": op["content"] or "",
        "length": op["length"] or 0,
    }
    doc = apply_op(doc, op_dict)
    preview = doc[:60] + '...' if len(doc) > 60 else doc
    print(f"  Op {op['id']:>2} by {op['user_name']:<10} → '{preview}'")

print(f"\n📄 Final document ({len(doc)} chars):")
print(doc)

## 🏗️ How the Server Uses OT

Here's the flow when two users edit simultaneously:

```
  Alice (Client A)              Server                  Bob (Client B)
       │                          │                          │
       │──INSERT(5, ", world")──►│                          │
       │                          │  1. Apply Alice's op     │
       │                          │  2. Store in operations  │
       │◄────── ACK ─────────────│                          │
       │                          │──broadcast to Bob───────►│
       │                          │                          │
       │                          │◄──DELETE(5, 1)──────────│
       │                          │  3. Transform Bob's op   │
       │                          │     against Alice's op   │
       │                          │  4. DELETE(5,1) becomes  │
       │                          │     DELETE(12,1)         │
       │                          │  5. Apply transformed op │
       │                          │  6. Store in operations  │
       │                          │──────── ACK ────────────►│
       │◄──broadcast to Alice────│                          │
       │                          │                          │
```

**Key insight**: The server is the single source of truth. All operations go through it, and it applies OT to maintain consistency.

In [ ]:
# Let's simulate the full server-side OT flow

class SimpleOTServer:
    """A minimal OT server that processes operations sequentially."""
    
    def __init__(self, initial_text=""):
        self.text = initial_text
        self.history = []  # all applied operations
        self.version = 0   # increments with each op
    
    def receive_op(self, op, client_version, user_name="Unknown"):
        """
        Receive an operation from a client.
        
        client_version: the version the client had when creating this op.
        If the client is behind, we transform the op against everything
        that happened since their version.
        """
        # Transform against all ops the client hasn't seen
        transformed = dict(op)
        ops_to_transform_against = self.history[client_version:]
        
        for past_op in ops_to_transform_against:
            transformed = transform(past_op, transformed)
        
        # Apply the transformed op
        old_text = self.text
        self.text = apply_op(self.text, transformed)
        self.history.append(transformed)
        self.version += 1
        
        print(f"  [{user_name}] version {client_version} → {self.version}")
        if op != transformed:
            orig = f"{op['type'].upper()}({op['position']})"
            trans = f"{transformed['type'].upper()}({transformed['position']})"
            print(f"    ⚡ Transformed: {orig} → {trans}")
        print(f"    Document: '{self.text}'")
        
        return transformed


# Simulate concurrent editing
print("🖥️  Simulating OT Server")
print("=" * 50)

server = SimpleOTServer("Hello!")
print(f"Initial document: '{server.text}' (version {server.version})")
print()

# Both Alice and Bob see "Hello!" at version 0
# Alice types ", world" at position 5 (version 0)
# Bob deletes "!" at position 5 (also version 0)

print("Alice sends INSERT(5, ', world') based on version 0:")
server.receive_op(make_insert(5, ", world"), client_version=0, user_name="Alice")

print("\nBob sends DELETE(5, 1) based on version 0:")
server.receive_op(make_delete(5, 1), client_version=0, user_name="Bob")

print(f"\n✅ Final document: '{server.text}'")
print(f"   Both intents preserved! Alice's comma and Bob's deletion both applied correctly.")

In [ ]:
# A more complex scenario: three users editing simultaneously

print("🖥️  Three-User OT Simulation")
print("=" * 50)

server = SimpleOTServer("The quick brown fox")
print(f"Initial: '{server.text}' (version {server.version})")
print()

# All three users see the same document at version 0
# Alice: insert " lazy" before "fox" (position 16)
# Bob: insert " jumps" at the end (position 19)
# Charlie: delete "quick " (position 4, length 6)

print("Alice inserts ' lazy' at position 16 (version 0):")
server.receive_op(make_insert(16, " lazy"), client_version=0, user_name="Alice")

print("\nBob inserts ' jumps' at position 19 (version 0):")
server.receive_op(make_insert(19, " jumps"), client_version=0, user_name="Bob")

print("\nCharlie deletes 'quick ' (pos 4, len 6) (version 0):")
server.receive_op(make_delete(4, 6), client_version=0, user_name="Charlie")

print(f"\n✅ Final: '{server.text}'")
print(f"\n💡 All three edits were applied correctly despite being concurrent!")
print(f"   - Alice's ' lazy' is before 'fox'")
print(f"   - Bob's ' jumps' is after the original 'fox' position")
print(f"   - Charlie's deletion of 'quick ' is applied")

## ⚠️ Challenges with OT

OT is powerful but comes with important trade-offs:

| Challenge | Description |
|-----------|-------------|
| **Central server required** | All operations must flow through a single server that maintains ordering |
| **Complex to implement correctly** | Edge cases with overlapping deletes, cursor tracking, rich text formatting |
| **Scaling limit** | One document = one server (Google Docs limits to ~100 concurrent editors) |
| **Latency sensitive** | Operations should be acknowledged within ~100ms for a good UX |

### Why Google Docs Chose OT

Despite these challenges, OT has important advantages:

- **Low memory**: only needs the current document + recent operations (not the full history)
- **Well-suited for text**: insert/delete operations map naturally to text editing
- **Battle-tested**: Google has used OT since 2006 (Google Wave → Google Docs)
- **100 editors is enough**: most real documents have 2-10 concurrent editors

## 🧹 Cleanup

In [ ]:
# Nothing to clean up — we only read from the database in this notebook
print("🧹 No cleanup needed — all operations were in-memory simulations.")

## 📚 Summary

### Key Takeaways

1. **Concurrent edits break documents** — applying the same operations in different orders gives different results
2. **OT transforms operations** — adjusting positions so each edit preserves the user's original intent
3. **The server is the authority** — all operations go through a central server that enforces ordering
4. **Operations are contextual** — each op was created against a specific document version
5. **Google Docs uses OT** — it's low memory, fast, and works great for ≤100 editors

### Next Up

In **Notebook 2**, we'll explore **CRDTs** — an alternative approach where operations can be applied in **any order** without a central server. This is what Figma and Apple Notes use.